## Monte Carlo Control example of Gymnasium Blackjack

In [ ]:
USE {
    repositories {
        mavenCentral()
        maven("https://central.sonatype.com/repository/maven-snapshots/")
    }
    dependencies {
        implementation("io.github.kotlinrl:integration:0.1.0-SNAPSHOT")
    }
}

In [ ]:
import io.github.kotlinrl.core.agent.*
import io.github.kotlinrl.core.learn.*
import io.github.kotlinrl.core.learn.mcc.*
import io.github.kotlinrl.core.policy.*
import io.github.kotlinrl.core.train.*
import io.github.kotlinrl.integration.gymnasium.*
import io.github.kotlinrl.integration.gymnasium.GymnasiumEnvs.*

Creating the following
- Env (BlackjackEnv = Env<List<Any>, Int, Tuple, Discrete> based on the Gymnasium structure)
- EpisodeCallback to log starting episodes
- StateActionListProvider to define the list of Actions for any State.  Blackjack only allows
    - Actions 1=Hit and 0=Stick
    - State is List<Any> based on the Tuple observation space = but really typed as List<Int>
        - We can use ```val (playerSum, dealerSum, usableAce) = observation.map { it as Int }``` to extract the data
- The QTable used to capture training information
    - Monte Carlo Control must wait until the end of each Episode to update the QTable

In [ ]:
val env = gymnasium.make<BlackjackEnv>(Blackjack_v1, seed = 123, render = true)

val episodeLogger = object : EpisodeCallback<List<Any>, Int> {
    override fun onEpisodeStart(episode: Int) {
        if (episode % 10_000 == 0) println("Starting episode $episode")
    }
}
val actionListProvider = StateActionListProvider<List<Any>, Int>{ listOf(0, 1) }

val qTable = QTable(
    shape = intArrayOf(32, 10, 2, 2),
    indexFor = { state: List<Any>, action: Int ->
        val (playerSum, dealerSum, usableAce) = state.map { it as Int }
        intArrayOf(playerSum, dealerSum - 1, usableAce, action)
    },
    actionListProvider = actionListProvider
)


Next we create the training Agent using the Monte Carlo Control
- We use an Epsilon Greedy Policy with a constant epsilon (rather than decaying epsilon) for the exploration factor
- The Epsilon Greedy Policy randomly chooses a number.
    - If it is less than the epsilon value it uses a Random Policy selection
    - Otherwise it uses a Greedy Policy to select the max q-value from the QTable

The Trainer uses the env and agent with a max steps per episode of 1 (this Env only allows for 1 hand per episode)
- We register the MonteCarloControl as a EpisodeCallback so that when the episode completes, it updates the QTable
- We also register the episode logger
- We then train for 100,000 episodes

In [ ]:
val trainingAgent = monteCarloAgent(
    id = "training",
    policy = epsilonGreedyPolicy(
        stateActionListProvider = actionListProvider,
        explorationFactor = constantEpsilon(0.1),
        qTable = qTable
    )
)
val trainer = trainer(
    env = env,
    agent = trainingAgent,
    maxStepsPerEpisode = 1,
    callbacks = listOf(
        MonteCarloControl(qTable, 0.99),
        episodeLogger
    )
)
println("Starting training")
val training = trainer.train(100_000)

Once training is complete, we create a testing agent
- No observations are consumed
- No learning alters the QTable.
- The Greedy Policy chooses the max Q-value from the QTable
    - It performs the best action given the state: (playerSum, dealerSum, usableAce)
We again test for 100,000 episodes to compare the episode results (i.e. the average reward achieved)

In [ ]:
val testingAgent = agent(
    id = "testing",
    policy = greedyPolicy(
        stateActionListProvider = actionListProvider,
        qTable = qTable
    )
)
val tester = trainer(
    env = env,
    agent = testingAgent,
    maxStepsPerEpisode = 1,
    callbacks = listOf(episodeLogger)
)
println("Starting testing")
val test = tester.train(100_000)

Comparing the average results:
We also check if the Agent has learned:
> - Q-value for [20, 10, 1, Stick]
> - Q-value for [20, 10, 1, Hit]:

This is a classic scenario:
> - Player has 20
> - Dealer shows 10
> - Usable ace is true
> - The Agent has learned that:
> - Sticking has a positive expected return (≈ 0.45)
> - Hitting has poor returns (possibly busted → 0.0)
> - ✅ That aligns with good Blackjack strategy: 20 is a strong hand — you usually stand.

In [ ]:
println("Training Results: ${training.episodeRewards.sum() / training.episodeRewards.size}")
println("Test Results: ${test.episodeRewards.sum() / test.episodeRewards.size}")

println("Q-value for [20, 10, 1, Stick]: ${qTable[listOf(20, 10, 1), 0]}")
println("Q-value for [20, 10, 1, Hit]:   ${qTable[listOf(20, 10, 1), 1]}")
